In [ ]:
import sys
sys.path.append('../..')  # preprocessing-mitgcm/, so the library packages can be imported

In [ ]:
import json
import os
import requests

def try_download(url):
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(r"Didn't work, response.status_code = " + str(response.status_code) + ", url = " + url)

    return response

def download_json(url: str) -> json:
    response = try_download(url)
    json_data = response.json()

    return json_data

def save_json(json_data: json, save_path) -> None:
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, 'w') as f:
        json.dump(json_data, f, indent=4)

In [ ]:
def download_bafu_from_api(station_id: int, variable_name: str, start_date: str, end_date: str, resampling_freq:str = None) -> json:
    resample_str = ""
    if resampling_freq:
        resample_str = f"?resample={resampling_freq}"

    url = (f"https://alplakes-internal-api.eawag.ch/bafu/hydrodata/measured/{station_id}/"
           f"{variable_name}/"
           f"{start_date}/"
           f"{end_date}"
           f"{resample_str}"
           )
    alplakes_depthtime_data = download_json(url)

    return alplakes_depthtime_data

In [ ]:
from lib.config import ConfigObject

# Rivers of the current project (config.json: RiverConfig[LakeName], requires "WithRivers": true)
config = ConfigObject('../../config.json')
bafu_folder = os.path.join(config.Paths.PreprocessingInputs, 'bafu_data')
year_to_download = 2025

In [ ]:
# Download one year of hourly discharge, month by month, for each river of the project
for river in config.Rivers:
    for i in range (1,13):
        try:
            start_download = f'{year_to_download}{str(i).zfill(2)}01'
            end_download = f'{year_to_download}{str(i+1).zfill(2)}01'
            if i == 12:
                end_download = f'{year_to_download+1}{str(1).zfill(2)}01'
            output_folder = os.path.join(bafu_folder, f'{river.Name}_{river.BafuId}')
            json_file = download_bafu_from_api(river.BafuId,
                                               river.DischargeVariableName,
                                               start_download,
                                               end_download,
                                               "hourly"
                                               )
            output_file_path = os.path.join(output_folder, f'{start_download}.json')
            save_json(json_file, output_file_path)

            print(f'Month {i} saved in {output_file_path}.')
        except Exception as e:
            print(f"Ouups, didn't work for month {i}: {e}")
            continue